# CS 521 HW3, Problem 3: Multi-Norm Adversarial Training

Use Google Colab with a GPU: **Runtime > Change runtime type > T4 GPU**.

All the code is in this notebook. Complete each place that has a `TODO` and `raise NotImplementedError`.
Do not change the data, the model, the attacks, the evaluation or the default hyperparameters,
unless the homework tells you to change them.

**Threat model.** For each test image $x$, the perturbation region is
$B_\infty(x, 8/255) \cup B_2(x, 0.5) \cup B_1(x, 12)$.
The union accuracy is the percentage of test images that are robust against all three attacks simultaneously.

# Set up for dataset and model

Download the base model, a subset of 10,000 CIFAR-10 training images (1,000 for each class),
and the first 1,000 CIFAR-10 test images from Google Drive.

In [ ]:
!gdown 17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes -O linf_base.pt
!gdown 1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe -O cifar10_train_10k.pt
!gdown 1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA -O cifar10_test_1k.pt

In [ ]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NORMS = ("linf", "l2", "l1")
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}

In [ ]:
def load_data(path):
    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.
    x, y = torch.load(path)
    return x.float().div(255).to(device), y.to(device)


# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.
x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

In [ ]:
# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


def load_base_model():
    model = make_model()
    model.load_state_dict(torch.load("linf_base.pt", map_location=device))
    return model

# Implement the Attacks

Each attack is untargeted, uses the cross-entropy loss, and returns `x_adv` with
$\|x_{adv} - x\|_p \leq \epsilon$ and $x_{adv} \in [0, 1]$. The attacks must not change the mode (`train` or `eval`) of the model.

* `pgd_linf`: start at a uniformly random point in the $\ell_\infty$ ball. For each step, add `alpha * sign(grad)`,
  clip the perturbation to $[-\epsilon, \epsilon]$, and clip the image to $[0, 1]$.
* `pgd_l2`: start at a random point in the $\ell_2$ ball. For each step, add `alpha * grad / ||grad||_2`,
  scale the perturbation back to the $\ell_2$ ball if its norm is larger than $\epsilon$, and clip the image to $[0, 1]$.

You can use your code from HW1. The $\ell_1$ attack (sparse $\ell_1$ descent, Tramer & Boneh 2019) is complete.

In [ ]:
def clip_box(x, delta):
    # Make x + delta an image in [0, 1]. This never increases |delta_i|.
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def l2_norm(v):
    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)


def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    delta = torch.empty_like(x).uniform_(-eps, eps) if rand_init else torch.zeros_like(x)
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = (delta + alpha * grad.sign()).clamp(-eps, eps)
        delta = clip_box(x, delta)
    return (x + delta).detach()


def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.randn_like(x)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / l2_norm(delta).clamp_min(1e-12) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = delta + alpha * grad / l2_norm(grad).clamp_min(1e-12)
        delta = delta * (eps / l2_norm(delta).clamp_min(eps))
        delta = clip_box(x, delta)
    return (x + delta).detach()

In [ ]:
def project_l1(delta, eps):
    # Projection of each image of delta onto the L1 ball of radius eps (Duchi et al., 2008).
    v = delta.flatten(1)
    a = v.abs()
    outside = a.sum(dim=1) > eps
    if not outside.any():
        return delta
    mu, _ = a.sort(dim=1, descending=True)
    cumsum = mu.cumsum(dim=1)
    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)
    rho = ((mu - (cumsum - eps) / idx > 0) * idx).argmax(dim=1, keepdim=True)
    theta = (cumsum.gather(1, rho) - eps) / (rho + 1).to(v.dtype)
    w = (a - theta).clamp_min(0) * v.sign()
    return torch.where(outside.unsqueeze(1), w, v).view(delta.shape)


def pgd_l1(model, x, y, eps, alpha, steps, rand_init=True, k=20):
    # Each step changes only the k coordinates with the largest gradient.
    # Coordinates at the image boundary that point outward are skipped.
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.distributions.Laplace(0.0, 1.0).sample(x.shape).to(x.device)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / delta.flatten(1).abs().sum(1).view(-1, 1, 1, 1) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y).flatten(1)
        x_cur = (x + delta).flatten(1)
        blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))
        grad = grad.masked_fill(blocked, 0)
        top = grad.abs().topk(k, dim=1).indices
        step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())
        delta = clip_box(x, delta + alpha * (step / k).view_as(x))
        delta = project_l1(delta, eps)
    return (x + delta).detach()


ATTACKS = {"linf": pgd_linf, "l2": pgd_l2, "l1": pgd_l1}


def attack(model, x, y, norm, steps=10):
    # The attack for one norm with the default eps and step size.
    alpha = EPS["l1"] / steps ** 0.5 if norm == "l1" else 2.5 * EPS[norm] / steps
    return ATTACKS[norm](model, x, y, EPS[norm], alpha, steps)

# Evaluate Single and Multi-Norm Robust Accuracy

`test_model` gives the clean accuracy, the robust accuracy for each norm, and the union accuracy (in %) on the
1,000 test images. An image is robust for a norm if the model is correct on the clean image and on the result
of each restart of the attack. An image is robust for the union if it is robust for all three norms simultaneously.
Use the default settings (`steps=50, restarts=2`) for all the numbers in your report.

In [ ]:
def test_model(model, steps=50, restarts=2, batch_size=250):
    model.eval()
    torch.manual_seed(0)  # the same random starts of the attacks for each model, so the results are comparable
    start = time.time()
    correct = {k: 0 for k in ("clean", *NORMS, "union")}
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        union = clean.clone()
        for norm in NORMS:
            robust = clean.clone()
            for _ in range(restarts):
                x_adv = attack(model, x, y, norm, steps)
                with torch.no_grad():
                    robust &= model(x_adv).argmax(1) == y
            correct[norm] += robust.sum().item()
            union &= robust
        correct["clean"] += clean.sum().item()
        correct["union"] += union.sum().item()
    acc = {k: 100.0 * v / len(y_test) for k, v in correct.items()}
    print("  ".join(f"{k} {v:.1f}" for k, v in acc.items()) + f"   ({time.time() - start:.0f} s)")
    return acc

# Adversarial Training

`adv_train(model, loss_fn, ...)` fine-tunes `model` for `epochs` epochs on the training subset.
For each batch, it calls `loss_fn(model, x, y, **kwargs)` and does one optimizer step on the loss.

A loss function makes the adversarial examples itself. Put the model in `eval` mode for the attack
(so that the attack does not change the BatchNorm statistics), and in `train` mode before you calculate the loss.

If `gp_beta` is set, `adv_train` does the gradient projection of RAMP (part (b)) before the optimizer step.

In [ ]:
def adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, gp_beta=None, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    params = list(model.parameters())
    start = time.time()
    for epoch in range(epochs):
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:  # skip the last, smaller batch
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            if gp_beta is None:
                loss.backward()
            else:
                # RAMP (part (b)): combine the adversarial gradient g_a and the natural gradient g_n
                # for each parameter tensor. gradient_projection is defined in part (b).
                g_a = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, ga, gn in zip(params, g_a, g_n):
                    p.grad = gp_beta * gradient_projection(gn, ga) + (1 - gp_beta) * ga
            opt.step()
            sched.step()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}: loss {np.mean(losses):.3f} ({time.time() - start:.0f} s)")
    model.eval()
    return model

In [ ]:
import json
import os

import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = "/content/drive/MyDrive/cs521_hw3_q3"
except Exception:
    SAVE_DIR = "q3_runs"
os.makedirs(SAVE_DIR, exist_ok=True)

RESULTS = {}
HISTORY = {}


def _path(name, ext):
    return os.path.join(SAVE_DIR, name.replace(" ", "_").replace("/", "-") + ext)


def run(name, loss_fn, seed=0, epochs=10, before=None, after=None, **kwargs):
    model = load_base_model()
    if os.path.exists(_path(name, ".json")):
        model.load_state_dict(torch.load(_path(name, ".pt"), map_location=device))
        saved = json.load(open(_path(name, ".json")))
        RESULTS[name] = saved["result"]
        if saved.get("history"):
            HISTORY[name] = saved["history"]
        print(f"{name}: loaded from {SAVE_DIR}")
        return model.eval()
    if before:
        before()
    start = time.time()
    adv_train(model, loss_fn, epochs=epochs, seed=seed, **kwargs)
    sec = (time.time() - start) / epochs
    history = after() if after else None
    RESULTS[name] = {**test_model(model), "s/epoch": sec}
    if history:
        HISTORY[name] = history
    torch.save(model.state_dict(), _path(name, ".pt"))
    json.dump({"result": RESULTS[name], "history": history}, open(_path(name, ".json"), "w"))
    return model


def show(names):
    return pd.DataFrame({n: RESULTS[n] for n in names}).T.round(1)


def mean_of(names, label):
    RESULTS[label] = pd.DataFrame({n: RESULTS[n] for n in names}).T.mean().to_dict()
    return RESULTS[label]

### Base model

In [ ]:
model_base = load_base_model()
RESULTS["base"] = {**test_model(model_base), "s/epoch": float("nan")}

# Part (a): Single-Norm Adversarial Training

Complete `single_norm_loss`: attack the batch with the norm `norm` (use `attack`), and return the
cross-entropy loss of the model on the adversarial images.

In [ ]:
def single_norm_loss(model, x, y, norm):
    model.eval()
    x_adv = attack(model, x, y, norm)
    model.train()
    return F.cross_entropy(model(x_adv), y)

In [ ]:
model_linf = run("linf-AT", single_norm_loss, norm="linf")

In [ ]:
model_l2 = run("l2-AT", single_norm_loss, norm="l2")

In [ ]:
model_l1 = run("l1-AT", single_norm_loss, norm="l1")

In [ ]:
show(["base", "linf-AT", "l2-AT", "l1-AT"])

# Part (b): Multi-Norm Adversarial Training (RAMP)

* `ramp_loss`: attack with $q = \ell_\infty$ and $r = \ell_1$. The loss is the mean of
  $\max(\mathcal{L}_{CE}(x^q), \mathcal{L}_{CE}(x^r))$ plus `lam` times $\mathcal{L}_{KL} = KL(p_q \| p_r)$.
  Calculate $\mathcal{L}_{KL}$ only on the images that the model classifies correctly under the $\ell_q$ attack.
* `gradient_projection(g_n, g_a)`: $\cos(g_n, g_a) \cdot g_n$ if $\cos(g_n, g_a) > 0$, else $0$.

`adv_train(..., gp_beta=0.5)` then uses `p.grad = gp_beta * GP(g_n, g_a) + (1 - gp_beta) * g_a` for each parameter
tensor, where $g_a$ is the gradient of `ramp_loss` and $g_n$ is the gradient of the cross-entropy loss on the clean images.

$\mathcal{L}_{KL}$ is averaged over the images that are correct under the $\ell_\infty$ attack (`batchmean` on that subset).

In [ ]:
def ramp_loss(model, x, y, lam=0.5):
    model.eval()
    x_q = attack(model, x, y, "linf")
    x_r = attack(model, x, y, "l1")
    model.train()
    logits_q = model(x_q)
    logits_r = model(x_r)
    ce = torch.max(F.cross_entropy(logits_q, y, reduction="none"), F.cross_entropy(logits_r, y, reduction="none")).mean()
    correct = logits_q.argmax(1) == y
    if not correct.any():
        return ce
    kl = F.kl_div(F.log_softmax(logits_r[correct], 1), F.log_softmax(logits_q[correct], 1),
                  reduction="batchmean", log_target=True)
    return ce + lam * kl


def gradient_projection(g_n, g_a):
    cos = F.cosine_similarity(g_n.flatten(), g_a.flatten(), dim=0)
    return cos.clamp_min(0) * g_n

In [ ]:
model_ramp = run("RAMP (seed 0)", ramp_loss, gp_beta=0.5)

In [ ]:
show(["base", "linf-AT", "l2-AT", "l1-AT", "RAMP (seed 0)"])

# Part (c): Error-Adaptive Mixed-Norm Training with a Base-Model Anchor

###(i) Each image is attacked with one norm, sampled from a distribution π over {ℓ∞, ℓ2, ℓ1}. π_p ∝ e_p + 0.05, where e_p is an EMA of the model’s training error on norm p (initialized at 0.5, momentum 0.9), and e_p is updated after each batch from the error on the images attacked with norm p. The loss is L = (1/n) Σ_i CE(f_θ(x′i), y_i) + μ · (1/|S|) Σ{i∈S} KL(p_{f₀}(x′i) ‖ p{f_θ}(x′_i)), where x′i = PGD{p_i}(x_i), f₀ is the frozen base model, S = {i : f₀(x′_i) = y_i}, and μ = 1. Union accuracy is limited by the weakest norm. Adaptive sampling puts more attacks on that norm, and each image costs only one attack, compared with two or three for MAX/RAMP. The anchor keeps the base model’s ℓ∞ robustness while the model learns ℓ1 robustness.

### (ii) Implementation

In [ ]:
TEACHER = None
MY_STATE = {}


def reset_my_state():
    global TEACHER
    if TEACHER is None:
        TEACHER = load_base_model().eval()
        for p in TEACHER.parameters():
            p.requires_grad_(False)
    MY_STATE.clear()
    MY_STATE.update(err=torch.full((len(NORMS),), 0.5, device=device), probs=[], errs=[], kl=[], kept=[])


def get_my_history():
    return {k: MY_STATE[k] for k in ("probs", "errs", "kl", "kept")}


def norm_probs(adaptive, floor=0.05):
    if not adaptive:
        return torch.full((len(NORMS),), 1 / len(NORMS), device=device)
    w = MY_STATE["err"] + floor
    return w / w.sum()


def my_loss(model, x, y, adaptive=True, mu=1.0, momentum=0.9):
    probs = norm_probs(adaptive)
    assign = torch.multinomial(probs, len(y), replacement=True)
    model.eval()
    x_adv = x.clone()
    for k, norm in enumerate(NORMS):
        m = assign == k
        if m.any():
            x_adv[m] = attack(model, x[m], y[m], norm)
    with torch.no_grad():
        t_logits = TEACHER(x_adv)
    model.train()
    logits = model(x_adv)
    loss = F.cross_entropy(logits, y)
    keep = t_logits.argmax(1) == y
    kl = torch.zeros((), device=x.device)
    if mu > 0 and keep.any():
        kl = F.kl_div(F.log_softmax(logits[keep], 1), F.log_softmax(t_logits[keep], 1),
                      reduction="batchmean", log_target=True)
        loss = loss + mu * kl
    with torch.no_grad():
        wrong = (logits.argmax(1) != y).float()
        for k in range(len(NORMS)):
            m = assign == k
            if m.any():
                MY_STATE["err"][k] = momentum * MY_STATE["err"][k] + (1 - momentum) * wrong[m].mean()
    MY_STATE["probs"].append(probs.tolist())
    MY_STATE["errs"].append(MY_STATE["err"].tolist())
    MY_STATE["kl"].append(kl.item())
    MY_STATE["kept"].append(keep.float().mean().item())
    return loss


def run_mine(name, seed=0, **kwargs):
    return run(name, my_loss, seed=seed, before=reset_my_state, after=get_my_history, **kwargs)

In [ ]:
model_mine0 = run_mine("Mine (seed 0)", seed=0)

In [ ]:
model_mine1 = run_mine("Mine (seed 1)", seed=1)

In [ ]:
model_ramp1 = run("RAMP (seed 1)", ramp_loss, seed=1, gp_beta=0.5)

### (iii) Results: mean of seeds 0 and 1

In [ ]:
mean_of(["RAMP (seed 0)", "RAMP (seed 1)"], "RAMP (mean)")
mean_of(["Mine (seed 0)", "Mine (seed 1)"], "Mine (mean)")
show(["base", "RAMP (seed 0)", "RAMP (seed 1)", "RAMP (mean)", "Mine (seed 0)", "Mine (seed 1)", "Mine (mean)"])

### (iv) Ablation

* **Uniform sampling:** keep the per-image mixed norms and the anchor, but use a fixed $\pi = (1/3, 1/3, 1/3)$.
* **No anchor** (optional, if time allows): keep the adaptive sampling, set $\mu = 0$.

In [ ]:
model_abl_uniform = run_mine("Ablation: uniform sampling", seed=0, adaptive=False)

In [ ]:
model_abl_noanchor = run_mine("Ablation: no anchor", seed=0, mu=0.0)

In [ ]:
show([n for n in ["Mine (seed 0)", "Ablation: uniform sampling", "Ablation: no anchor"] if n in RESULTS])

### (v) Analysis

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
h = HISTORY["Mine (seed 0)"]
steps = range(len(h["probs"]))
for k, norm in enumerate(NORMS):
    axes[0].plot(steps, [p[k] for p in h["probs"]], label=norm)
    axes[1].plot(steps, [e[k] for e in h["errs"]], label=norm)
axes[0].set_title("sampling probability per norm")
axes[1].set_title("running training error per norm (EMA)")
axes[2].plot(steps, h["kl"], label="KL to base model")
axes[2].plot(steps, h["kept"], label="fraction distilled (teacher correct)")
axes[2].set_title("anchor term")
for ax in axes:
    ax.set_xlabel("training step")
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

n = len(h["probs"])
w = max(1, n // 10)
for name, part in [("first 10%", slice(0, w)), ("last 10%", slice(n - w, n))]:
    p = np.array(h["probs"][part]).mean(0)
    print(f"{name}: mean sampling probs " + "  ".join(f"{norm} {v:.2f}" for norm, v in zip(NORMS, p)))

### (vi) Check the evaluation

Stronger attacks (200 steps, 5 restarts) on the best model. The method does not change the model at test time.


In [ ]:
best_name = max(["Mine (seed 0)", "Mine (seed 1)", "RAMP (seed 0)", "RAMP (seed 1)"], key=lambda n: RESULTS[n]["union"])
best_model = {"Mine (seed 0)": model_mine0, "Mine (seed 1)": model_mine1,
              "RAMP (seed 0)": model_ramp, "RAMP (seed 1)": model_ramp1}[best_name]
print("best model:", best_name)
strong = test_model(best_model, steps=200, restarts=5)
pd.DataFrame({"default (50 steps, 2 restarts)": RESULTS[best_name], "strong (200 steps, 5 restarts)": strong}).T.drop(columns="s/epoch", errors="ignore").round(1)

### Summary table

In [ ]:
show(["base", "linf-AT", "l2-AT", "l1-AT", "RAMP (mean)", "Mine (mean)"]
     + [n for n in ["Ablation: uniform sampling", "Ablation: no anchor"] if n in RESULTS])